In [3]:
import mujoco
from mujoco import mjx
import jax
import jax.numpy as jnp

In [2]:
print("MuJoCo version:", mujoco.__version__)
print("MJX imported successfully")

MuJoCo version: 3.8.1
MJX imported successfully


In [17]:
# path to the s MuJoCo model
from pathlib import Path

path = Path('universal_robots_ur5e/scene.xml')
if path.exists():
    print("Path exists")
else:
    print("Path does not exist")

Path exists


In [18]:
# Load the MuJoCo model from an XML file and initialize the simulation state (CPU)
mj_model = mujoco.MjModel.from_xml_path(str(path))
mj_data = mujoco.MjData(mj_model)

# Transfer the MuJoCo model and data to MJX for hardware-accelerated simulation (GPU/TPU)
mjx_model = mjx.put_model(mj_model)
mjx_data = mjx.put_data(mj_model, mj_data)

In [19]:
# This function answers:If the joints are q, where is the robot hand?
# this code is copied from assignment

def forward_kinematics(q):

    new_mjx_data = mjx_data.replace(qpos=q)

    new_mjx_data = mjx.fwd_position(
        mjx_model,
        new_mjx_data
    )

    pos = new_mjx_data.site_xpos[
        mj_model.site('attachment_site').id
    ]

    mat = new_mjx_data.site_xmat[
        mj_model.site('attachment_site').id
    ]

    return pos, mat

In [20]:
## test the FK
print(forward_kinematics(jnp.zeros(6)))

(Array([-0.81700003, -0.234     ,  0.06300008], dtype=float32), Array([[ 0.9999998,  0.       , -0.       ],
       [-0.       ,  0.       , -0.9999998],
       [ 0.       ,  0.9999998,  0.       ]], dtype=float32))


In [21]:
print(forward_kinematics(jnp.ones(6)))

(Array([ 0.17492712, -0.0755766 , -0.46394524], dtype=float32), Array([[ 0.16226359, -0.39383033,  0.90474766],
       [-0.58876044,  0.69715893,  0.40906072],
       [-0.79185337, -0.5990553 , -0.1187482 ]], dtype=float32))
